In [1]:
from pathlib import Path
import pandas as pd

# Set the processed datasets that will feed the final ward master table.
# Group 1 is the spine, while Groups 2 and 3 provide context around each ward-year record.

base_dir = Path(
    r"C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed"
)

group1_path = base_dir / "01_ward_election_panel" / "ward_election_panel_2011_2021.csv"
group2_path = base_dir / "02_province_context_panel" / "province_context_panel_2011_2023.csv"
group3_path = base_dir / "03_municipality_context" / "municipality_context.csv"

# Group 4 was excluded because the source was not genuinely ward-level.
# Groups 5 and 6 remain outside this master table by design.

group4_path = base_dir / "04_ward_economic_context" / "ward_economic_context_2022.csv"

output_dir = base_dir / "07_ward_master_dataset"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "ward_master_dataset_2011_2021.csv"

print("Group 1 exists:", group1_path.exists())
print("Group 2 exists:", group2_path.exists())
print("Group 3 exists:", group3_path.exists())
print("Group 4 exists:", group4_path.exists())
print("Output folder:", output_dir)

Group 1 exists: True
Group 2 exists: True
Group 3 exists: True
Group 4 exists: False
Output folder: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\07_ward_master_dataset


In [2]:
# Load the three validated datasets that form the final ward master table.
# We keep Group 1 as the starting spine so its ward-year records are never replaced.

ward_panel = pd.read_csv(group1_path)
province_context = pd.read_csv(group2_path)
municipality_context = pd.read_csv(group3_path)

print("Group 1 shape:", ward_panel.shape)
print("Group 2 shape:", province_context.shape)
print("Group 3 shape:", municipality_context.shape)

print("\nGroup 1 columns:")
print(ward_panel.columns.tolist())

print("\nGroup 2 columns:")
print(province_context.columns.tolist())

print("\nGroup 3 columns:")
print(municipality_context.columns.tolist())

Group 1 shape: (2599, 11)
Group 2 shape: (3, 27)
Group 3 shape: (44, 10)

Group 1 columns:
['Province', 'Municipality', 'Ward', 'RegisteredVoters', 'SpoiltVotes', 'TotalValidVotes', 'VotingDistricts', 'VotingStations', 'ElectionYear', 'TurnoutRate', 'BoundaryConsistent']

Group 2 columns:
['geography_level', 'geography_name', 'year', 'Food poverty headcount (FPL, %)', 'Gini coefficient (income per capita)', 'Poverty gap (P1, %)', 'Poverty headcount (P0, %)', 'Poverty share (%)', 'Severity of poverty (P2, %)', 'Dissatisfied with democracy', 'Distrust of national government', 'Distrust of political parties', 'Distrust of provincial government', 'Distrust of the IEC', 'QLFS_Q1_2024_Snapshot', 'qlfs_geography', 'QLFS_Q1_2024_DiscouragedWorkSeekers', 'QLFS_Q1_2024_Employed_thousand', 'QLFS_Q1_2024_AbsorptionRate', 'QLFS_Q1_2024_LabourForce_thousand', 'QLFS_Q1_2024_LabourForceParticipationRate', 'QLFS_Q1_2024_NotEconomicallyActive_thousand', 'QLFS_Q1_2024_OtherNotEconomicallyActive_thousand'

In [3]:
# Inspect the geographic and year keys before joining the context datasets.
# This makes sure the names and years can be aligned without creating incorrect matches.

print("Group 1 provinces:")
print(ward_panel["Province"].unique())

print("\nGroup 1 election years:")
print(sorted(ward_panel["ElectionYear"].unique()))

print("\nGroup 1 municipalities:", ward_panel["Municipality"].nunique())

print("\nGroup 2 geography levels:")
print(province_context["geography_level"].unique())

print("\nGroup 2 geography names:")
print(province_context["geography_name"].unique())

print("\nGroup 2 years:")
print(sorted(province_context["year"].unique()))

print("\nGroup 3 municipalities:", municipality_context["Municipality"].nunique())

Group 1 provinces:
['KwaZulu-Natal']

Group 1 election years:
[np.int64(2011), np.int64(2016), np.int64(2021)]

Group 1 municipalities: 114

Group 2 geography levels:
['Province']

Group 2 geography names:
['KwaZulu-Natal']

Group 2 years:
[np.int64(2011), np.int64(2015), np.int64(2023)]

Group 3 municipalities: 44


This gives us an important finding before the merge. Group 1 contains KZN ward-level data for 2011, 2016, and 2021, Group 2 contains KZN province-level data for 2011, 2015, and 2023, and Group 3 contains KZN municipality-level data without a year. Because Group 2 does not contain exact 2016 or 2021 values, we will apply the planned nearest-year rule: 2011 will use 2011, 2016 will use 2015, and 2021 will use 2023. This means that each ward-year in the same election year will receive the corresponding KZN province-level context, which matches the province-level grain of Group 2. We also found that Group 1 has 114 unique municipality names while Group 3 has 44. This does not necessarily indicate an error because municipality names can be repeated across election years and may reflect historical naming or boundary changes. We will therefore not blindly merge Group 3 yet and will first check how the municipality names and boundaries correspond between the two datasets.


In [4]:
# Compare municipality names between the ward spine and municipality context.
# This identifies historical naming or boundary differences before the final municipality join.

ward_municipalities = set(
    ward_panel["Municipality"].dropna().astype(str).str.strip()
)

context_municipalities = set(
    municipality_context["Municipality"].dropna().astype(str).str.strip()
)

unmatched_municipalities = sorted(
    ward_municipalities - context_municipalities
)

unused_context_municipalities = sorted(
    context_municipalities - ward_municipalities
)

print("Group 1 unique municipalities:", len(ward_municipalities))
print("Group 3 unique municipalities:", len(context_municipalities))

print("\nGroup 1 municipalities without an exact Group 3 match:",
      len(unmatched_municipalities))

for municipality in unmatched_municipalities:
    print(municipality)

print("\nGroup 3 municipalities not found in Group 1:",
      len(unused_context_municipalities))

for municipality in unused_context_municipalities:
    print(municipality)

Group 1 unique municipalities: 114
Group 3 unique municipalities: 44

Group 1 municipalities without an exact Group 3 match: 114
ETH - eThekwini
ETH - eThekwini [Durban Metro]
KZN211 - Vulamehlo [Dududu]
KZN212 - Umdoni
KZN212 - Umdoni [Scottburgh]
KZN212 - uMdoni
KZN213 - Umzumbe
KZN213 - Umzumbe [Umzumbe]
KZN213 - uMzumbe
KZN214 - UMuziwabantu
KZN214 - UMuziwabantu [Harding]
KZN214 - uMuziwabantu
KZN215 - Ezinqoleni [Izinqolweni]
KZN216 - Hibiscus Coast
KZN216 - Hibiscus Coast [Port Shepstone]
KZN216 - Ray Nkonyeni
KZN221 - uMshwathi
KZN221 - uMshwathi [Wartburg]
KZN222 - uMngeni
KZN222 - uMngeni [Howick]
KZN223 - Mooi Mpofana
KZN223 - Mooi Mpofana [Mooirivier]
KZN223 - Mpofana
KZN224 - Impendle
KZN224 - Impendle [Impendle]
KZN224 - iMpendle
KZN225 - Msunduzi
KZN225 - Msunduzi [Pietermaritzburg]
KZN226 - Mkhambathini
KZN226 - Mkhambathini [Camperdown]
KZN227 - Richmond
KZN227 - Richmond [Richmond]
KZN232 - Emnambithi/Ladysmith [Ladysmith]
KZN233 - Indaka [Waaihoek]
KZN234 - Umtshezi 

This is why we checked the keys before joining. Group 1 uses historical IEC municipality labels with municipality codes and, in some cases, local-area names, such as `KZN212 - Umdoni`, `KZN212 - Umdoni [Scottburgh]`, and `KZN212 - uMdoni`. Group 3 uses current formal municipality names, such as `Umdoni Local Municipality`, `Ethekwini Metropolitan Municipality`, and `The Msunduzi Local Municipality`. An exact string join would therefore produce zero municipality matches, even though many of the municipalities are substantively the same. Group 1 also contains historical municipalities that no longer exist in the same form, including Vulamehlo, Ezinqoleni, and Hibiscus Coast. We will not blindly map these historical municipalities to current municipalities because this could assign incorrect context. We will use the municipality codes embedded in Group 1 where they are available and first examine the 44 current municipalities in Group 3 to determine whether municipality codes are available elsewhere in the dataset. We will then construct a controlled mapping only for municipalities that can be safely matched.


In [5]:
# Extract the IEC municipality code from the historical ward-election labels.
# The code gives us a more reliable key than trying to match changing municipality names.

ward_panel["MunicipalityCode"] = (
    ward_panel["Municipality"]
    .astype(str)
    .str.extract(r"^(KZN\d+|ETH)", expand=False)
)

print("Unique municipality codes:", ward_panel["MunicipalityCode"].nunique())
print("\nMunicipality codes found:")

print(
    sorted(
        ward_panel["MunicipalityCode"]
        .dropna()
        .unique()
    )
)

Unique municipality codes: 55

Municipality codes found:
['ETH', 'KZN211', 'KZN212', 'KZN213', 'KZN214', 'KZN215', 'KZN216', 'KZN221', 'KZN222', 'KZN223', 'KZN224', 'KZN225', 'KZN226', 'KZN227', 'KZN232', 'KZN233', 'KZN234', 'KZN235', 'KZN236', 'KZN237', 'KZN238', 'KZN241', 'KZN242', 'KZN244', 'KZN245', 'KZN252', 'KZN253', 'KZN254', 'KZN261', 'KZN262', 'KZN263', 'KZN265', 'KZN266', 'KZN271', 'KZN272', 'KZN273', 'KZN274', 'KZN275', 'KZN276', 'KZN281', 'KZN282', 'KZN283', 'KZN284', 'KZN285', 'KZN286', 'KZN291', 'KZN292', 'KZN293', 'KZN294', 'KZN431', 'KZN432', 'KZN433', 'KZN434', 'KZN435', 'KZN436']


That confirms the situation clearly. 55 historical municipality codes appear in Group 1, while only 44 current municipalities exist in Group 3.

The difference is the historical municipalities that were later consolidated/reconfigured. We should not map those 11 historical codes to current municipalities unless the mapping is explicitly justified.

For the final master, the safest approach is:

Keep the Group 1 historical municipality label and ward-year structure intact.
Attach Group 3 context only where the municipality corresponds to one of the 44 current municipalities.
Do not force the 11 historical-only municipality codes into current municipality context.
Record unmatched municipality-context records as a limitation rather than manufacturing a match.

First, let's construct the controlled mapping for the 44 current municipalities and validate it against Group 3.

In [7]:
# Map the 44 current municipality context records to their corresponding IEC municipality codes.
# Historical-only municipalities remain unmatched rather than receiving potentially incorrect context.

municipality_code_map = {
    "ETH": "Ethekwini Metropolitan Municipality",
    "KZN212": "Umdoni Local Municipality",
    "KZN213": "Umzumbe Local Municipality",
    "KZN214": "UMuziwabantu Local Municipality",
    "KZN216": "Ray Nkonyeni Local Municipality",
    "KZN221": "uMshwathi Local Municipality",
    "KZN222": "uMngeni Local Municipality",
    "KZN223": "Mpofana Local Municipality",
    "KZN224": "Impendle Local Municipality",
    "KZN225": "The Msunduzi Local Municipality",
    "KZN226": "Mkhambathini Local Municipality",
    "KZN227": "Richmond Local Municipality",
    "KZN235": "Okhahlamba Local Municipality",
    "KZN237": "Inkosi Langalibalele Local Municipality",
    "KZN238": "Alfred Duma Local Municipality",
    "KZN241": "Endumeni Local Municipality",
    "KZN242": "Nqutu Local Municipality",
    "KZN244": "Msinga Local Municipality",
    "KZN245": "Umvoti Local Municipality",
    "KZN252": "Newcastle Local Municipality",
    "KZN253": "Emadlangeni Local Municipality",
    "KZN254": "Dannhauser Local Municipality",
    "KZN261": "eDumbe Local Municipality",
    "KZN262": "UPhongolo Local Municipality",
    "KZN263": "Abaqulusi Local Municipality",
    "KZN265": "Nongoma Local Municipality",
    "KZN266": "Ulundi Local Municipality",
    "KZN272": "Jozini Local Municipality",
    "KZN275": "Mtubatuba Local Municipality",
    "KZN276": "Big Five Hlabisa Local Municipality",
    "KZN281": "Mfolozi Local Municipality",
    "KZN282": "uMhlathuze Local Municipality",
    "KZN284": "uMlalazi Local Municipality",
    "KZN285": "Mthonjaneni Local Municipality",
    "KZN286": "Nkandla Local Municipality",
    "KZN291": "Mandeni Local Municipality",
    "KZN292": "KwaDukuza Local Municipality",
    "KZN293": "Ndwedwe Local Municipality",
    "KZN294": "Maphumulo Local Municipality",
    "KZN433": "Greater Kokstad Local Municipality",
    "KZN434": "Ubuhlebezwe Local Municipality",
    "KZN435": "Umzimkhulu Local Municipality",
    "KZN436": "Dr Nkosazana Dlamini Zuma Local Municipality"
}

group3_expected = set(
    municipality_code_map.values()
)

group3_actual = set(
    municipality_context["Municipality"].astype(str).str.strip()
)

missing_group3 = sorted(group3_expected - group3_actual)
extra_group3 = sorted(group3_actual - group3_expected)

print("Mapped municipality codes:", len(municipality_code_map))
print("Expected current municipalities:", len(group3_expected))
print("Group 3 municipalities:", len(group3_actual))

print("\nExpected municipalities missing from Group 3:", len(missing_group3))
for municipality in missing_group3:
    print(municipality)

print("\nUnexpected Group 3 municipalities:", len(extra_group3))
for municipality in extra_group3:
    print(municipality)

if (
    len(municipality_code_map) == 44
    and len(missing_group3) == 0
    and len(extra_group3) == 0
):
    print("\nMunicipality mapping validation: PASSED")
else:
    print("\nMunicipality mapping validation: REVIEW REQUIRED")

Mapped municipality codes: 43
Expected current municipalities: 43
Group 3 municipalities: 44

Expected municipalities missing from Group 3: 0

Unexpected Group 3 municipalities: 1
Umhlabuyalingana Local Municipality

Municipality mapping validation: REVIEW REQUIRED


#=============================================================================#


Good  this caught exactly the kind of mismatch we want to catch before merging.

The issue is that our mapping has 43 codes, not 44. The missing current municipality is:

Umhlabuyalingana Local Municipality

And importantly, Group 3 itself has all 44 municipalities, so we should add the historical IEC code for Umhlabuyalingana rather than treating Group 3 as wrong.

For the next cell, let's identify the exact code already present in Group 1 for that municipality before adding it to the mapping.

In [8]:
# Check the historical labels for Umhlabuyalingana before adding its current municipality mapping.
# This lets us use the actual IEC code from our data instead of guessing the join key.

umhlabuyalingana_rows = ward_panel[
    ward_panel["Municipality"].astype(str).str.contains(
        "Umhlabuyalingana|uMhlabuyalingana",
        case=False,
        na=False
    )
]

print("Historical Umhlabuyalingana labels:")
print(
    umhlabuyalingana_rows["Municipality"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("\nIEC codes found:")
print(
    umhlabuyalingana_rows["MunicipalityCode"]
    .dropna()
    .unique()
)

Historical Umhlabuyalingana labels:
['KZN271 - Umhlabuyalingana', 'KZN271 - Umhlabuyalingana [Emangusi]', 'KZN271 - uMhlabuyalingana']

IEC codes found:
['KZN271']


Perfect. We now have the exact historical IEC code: KZN271.

So we can safely add that to the mapping. After that, we should validate that all 44 current municipalities are mapped exactly once, with no unexpected municipalities.

In [10]:
# Add the verified IEC code for Umhlabuyalingana to complete the current municipality mapping.
# This gives us a controlled bridge between historical ward labels and current municipality context.

municipality_code_map["KZN271"] = "Umhlabuyalingana Local Municipality"

group3_expected = set(
    municipality_code_map.values()
)

group3_actual = set(
    municipality_context["Municipality"].astype(str).str.strip()
)

missing_group3 = sorted(group3_expected - group3_actual)
extra_group3 = sorted(group3_actual - group3_expected)

print("Mapped municipality codes:", len(municipality_code_map))
print("Expected current municipalities:", len(group3_expected))
print("Group 3 municipalities:", len(group3_actual))

print("\nExpected municipalities missing from Group 3:", len(missing_group3))
for municipality in missing_group3:
    print(municipality)

print("\nUnexpected Group 3 municipalities:", len(extra_group3))
for municipality in extra_group3:
    print(municipality)

if (
    len(municipality_code_map) == 44
    and len(group3_expected) == 44
    and len(group3_actual) == 44
    and len(missing_group3) == 0
    and len(extra_group3) == 0
):
    print("\nMunicipality mapping validation: PASSED")
else:
    print("\nMunicipality mapping validation: REVIEW REQUIRED")

Mapped municipality codes: 44
Expected current municipalities: 44
Group 3 municipalities: 44

Expected municipalities missing from Group 3: 0

Unexpected Group 3 municipalities: 0

Municipality mapping validation: PASSED


In [12]:
# Create the current municipality name from the validated IEC code mapping.
# Historical-only municipalities will stay unmatched instead of receiving incorrect current context.

ward_panel["CurrentMunicipality"] = (
    ward_panel["MunicipalityCode"]
    .map(municipality_code_map)
)

print("Ward-year rows:", len(ward_panel))
print("Current municipality matches:", ward_panel["CurrentMunicipality"].notna().sum())
print("Current municipality unmatched:", ward_panel["CurrentMunicipality"].isna().sum())

print("\nHistorical-only municipality codes:")
print(
    sorted(
        ward_panel.loc[
            ward_panel["CurrentMunicipality"].isna(),
            "MunicipalityCode"
        ].dropna().unique()
    )
)

Ward-year rows: 2599
Current municipality matches: 2489
Current municipality unmatched: 110

Historical-only municipality codes:
['KZN211', 'KZN215', 'KZN232', 'KZN233', 'KZN234', 'KZN236', 'KZN273', 'KZN274', 'KZN283', 'KZN431', 'KZN432']


Perfect. Cell 09 gives us the expected result.

2,599 ward-year records remain on the Group 1 spine.
2,489 can receive current municipality context.
110 records belong to the 11 historical-only municipality codes.
We will not force those 110 records into a current municipality. That preserves data integrity.

Next, before the actual merge, we should verify that Group 3 has exactly one row per current municipality. This protects us from accidentally multiplying the 2,599 spine rows during the merge.

In [14]:
# Check that each current municipality has exactly one context record before the merge.
# A one-to-one municipality context prevents the ward-year spine from gaining duplicate rows.

group3_join_check = (
    municipality_context
    .assign(Municipality=municipality_context["Municipality"].astype(str).str.strip())
    .groupby("Municipality")
    .size()
)

print("Group 3 rows:", len(municipality_context))
print("Unique municipality keys:", group3_join_check.index.nunique())
print("Municipalities with duplicate context rows:", (group3_join_check > 1).sum())

if (
    len(municipality_context) == 44
    and group3_join_check.index.nunique() == 44
    and (group3_join_check > 1).sum() == 0
):
    print("\nGroup 3 join-key validation: PASSED")
else:
    print("\nGroup 3 join-key validation: REVIEW REQUIRED")

Group 3 rows: 44
Unique municipality keys: 44
Municipalities with duplicate context rows: 0

Group 3 join-key validation: PASSED


In [15]:
# Join current municipality context onto the historical ward-year spine.
# The left join keeps every Group 1 ward-year record, including historical-only municipalities.

municipality_context_join = municipality_context.copy()

municipality_context_join["Municipality"] = (
    municipality_context_join["Municipality"]
    .astype(str)
    .str.strip()
)

ward_master = ward_panel.merge(
    municipality_context_join,
    left_on="CurrentMunicipality",
    right_on="Municipality",
    how="left",
    validate="many_to_one",
    suffixes=("", "_Group3")
)

print("Group 1 rows:", len(ward_panel))
print("Master rows after Group 3 merge:", len(ward_master))
print("Rows added or lost:", len(ward_master) - len(ward_panel))

if len(ward_master) == len(ward_panel):
    print("\nGroup 3 merge row-count validation: PASSED")
else:
    print("\nGroup 3 merge row-count validation: FAILED")

Group 1 rows: 2599
Master rows after Group 3 merge: 2599
Rows added or lost: 0

Group 3 merge row-count validation: PASSED


In [16]:
# Check which ward-year records received municipality context after the merge.
# We also confirm the original boundary consistency field is still present in the master table.

context_columns = [
    "Household",
    "Homeless",
    "Transient",
    "Institution",
    "UrbanArea",
    "TribalOrTraditionalArea",
    "FarmArea",
    "MalePopulation",
    "Population"
]

context_received = ward_master[context_columns].notna().any(axis=1)

print("Master rows:", len(ward_master))
print("Rows with municipality context:", context_received.sum())
print("Rows without municipality context:", (~context_received).sum())

print("\nBoundaryConsistent present:", "BoundaryConsistent" in ward_master.columns)
print("BoundaryConsistent missing values:", ward_master["BoundaryConsistent"].isna().sum())

if (
    len(ward_master) == 2599
    and context_received.sum() == 2489
    and (~context_received).sum() == 110
    and "BoundaryConsistent" in ward_master.columns
):
    print("\nMunicipality merge coverage validation: PASSED")
else:
    print("\nMunicipality merge coverage validation: REVIEW REQUIRED")

Master rows: 2599
Rows with municipality context: 2489
Rows without municipality context: 110

BoundaryConsistent present: True
BoundaryConsistent missing values: 0

Municipality merge coverage validation: PASSED


#==============================================================================
#==============================================================================

We now have a clean municipality merge:

2,599 total ward-year rows retained.
2,489 received municipality context.
110 historical-only rows intentionally remain without current municipality context.
BoundaryConsistent survived with 0 missing values.

Now we move to Group 2  province context. This needs a year alignment because Group 2 has 2011, 2015, 2023, while the election spine has 2011, 2016, 2021.

We will use the nearest available context year:

2011 to 2011

2016 to 2015

2021 to 023

But first, let's create and validate that year mapping explicitly.

In [17]:
# Map each election year to the nearest available province-context year.
# This keeps the election spine unchanged while using the closest available socioeconomic context.

province_year_map = {
    2011: 2011,
    2016: 2015,
    2021: 2023
}

ward_master["ProvinceContextYear"] = (
    ward_master["ElectionYear"]
    .map(province_year_map)
)

print("Election years:", sorted(ward_master["ElectionYear"].unique()))
print("Mapped context years:", sorted(ward_master["ProvinceContextYear"].unique()))

print("\nElection-to-context year mapping:")
for election_year, context_year in province_year_map.items():
    print(f"{election_year} -> {context_year}")

print("\nMissing province context-year keys:",
      ward_master["ProvinceContextYear"].isna().sum())

if (
    ward_master["ProvinceContextYear"].isna().sum() == 0
    and set(ward_master["ProvinceContextYear"].unique()) == {2011, 2015, 2023}
):
    print("\nProvince context year mapping: PASSED")
else:
    print("\nProvince context year mapping: REVIEW REQUIRED")

Election years: [np.int64(2011), np.int64(2016), np.int64(2021)]
Mapped context years: [np.int64(2011), np.int64(2015), np.int64(2023)]

Election-to-context year mapping:
2011 -> 2011
2016 -> 2015
2021 -> 2023

Missing province context-year keys: 0

Province context year mapping: PASSED


In [18]:
# Check that Group 2 has exactly one KwaZulu-Natal context record for each available year.
# A unique province-year key keeps the final ward master table from gaining duplicate rows.

province_join_check = (
    province_context[
        ["geography_name", "year"]
    ]
    .astype({"geography_name": str})
    .groupby(["geography_name", "year"])
    .size()
)

print("Group 2 rows:", len(province_context))
print("Unique province-year keys:", province_join_check.index.nunique())
print("Duplicate province-year keys:", (province_join_check > 1).sum())

print("\nProvince-context years:")
print(sorted(province_context["year"].unique()))

if (
    len(province_context) == 3
    and province_join_check.index.nunique() == 3
    and (province_join_check > 1).sum() == 0
):
    print("\nProvince context join-key validation: PASSED")
else:
    print("\nProvince context join-key validation: REVIEW REQUIRED")

Group 2 rows: 3
Unique province-year keys: 3
Duplicate province-year keys: 0

Province-context years:
[np.int64(2011), np.int64(2015), np.int64(2023)]

Province context join-key validation: PASSED


In [19]:
# Add the nearest available province-level context to each ward-year record.
# The left join keeps the 2,599-row master spine unchanged.

province_context_join = province_context.copy()

province_context_join["geography_name"] = (
    province_context_join["geography_name"]
    .astype(str)
    .str.strip()
)

ward_master = ward_master.merge(
    province_context_join,
    left_on=["Province", "ProvinceContextYear"],
    right_on=["geography_name", "year"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_Group2")
)

print("Rows before Group 2 merge:", 2599)
print("Master rows after Group 2 merge:", len(ward_master))
print("Rows added or lost:", len(ward_master) - 2599)

if len(ward_master) == 2599:
    print("\nGroup 2 merge row-count validation: PASSED")
else:
    print("\nGroup 2 merge row-count validation: FAILED")

Rows before Group 2 merge: 2599
Master rows after Group 2 merge: 2599
Rows added or lost: 0

Group 2 merge row-count validation: PASSED


In [20]:
# Check that every ward-year received the expected province context after the merge.
# This confirms the nearest-year join worked without leaving gaps in the KZN master table.

province_context_received = ward_master["geography_name"].notna()

print("Master rows:", len(ward_master))
print("Rows with province context:", province_context_received.sum())
print("Rows without province context:", (~province_context_received).sum())

print("\nProvince context years used:")
print(
    ward_master[
        ["ElectionYear", "ProvinceContextYear"]
    ]
    .drop_duplicates()
    .sort_values("ElectionYear")
    .to_string(index=False)
)

print("\nBoundaryConsistent present:", "BoundaryConsistent" in ward_master.columns)
print("BoundaryConsistent missing values:",
      ward_master["BoundaryConsistent"].isna().sum())

if (
    len(ward_master) == 2599
    and province_context_received.sum() == 2599
    and (~province_context_received).sum() == 0
    and "BoundaryConsistent" in ward_master.columns
):
    print("\nProvince merge coverage validation: PASSED")
else:
    print("\nProvince merge coverage validation: REVIEW REQUIRED")

Master rows: 2599
Rows with province context: 2599
Rows without province context: 0

Province context years used:
 ElectionYear  ProvinceContextYear
         2011                 2011
         2016                 2015
         2021                 2023

BoundaryConsistent present: True
BoundaryConsistent missing values: 0

Province merge coverage validation: PASSED


Current master status
Check	Result
Ward-year spine	2,599 rows
Municipality context matched	2,489
Historical-only municipality rows	110
Province context matched	2,599
BoundaryConsistent preserved	Yes
Province year mapping	2011→2011, 2016→2015, 2021→2023
Row multiplication	None

Now we should do a final structural inspection before saving. This lets us catch duplicate columns, unexpected keys, missing values, and confirm the final shape.

In [21]:
# Inspect the completed master table before saving it to the processed folder.
# This final check confirms the ward-year spine and merged context are still structurally sound.

print("Final master shape:", ward_master.shape)

print("\nFinal master columns:")
print(ward_master.columns.tolist())

print("\nDuplicate rows:", ward_master.duplicated().sum())

print("\nWard-year duplicate keys:")
print(
    ward_master.duplicated(
        subset=["Municipality", "Ward", "ElectionYear"]
    ).sum()
)

print("\nMissing values in key fields:")
print(
    ward_master[
        ["Province", "Municipality", "Ward", "ElectionYear", "TurnoutRate"]
    ]
    .isna()
    .sum()
)

Final master shape: (2599, 51)

Final master columns:
['Province', 'Municipality', 'Ward', 'RegisteredVoters', 'SpoiltVotes', 'TotalValidVotes', 'VotingDistricts', 'VotingStations', 'ElectionYear', 'TurnoutRate', 'BoundaryConsistent', 'MunicipalityCode', 'CurrentMunicipality', 'Municipality_Group3', 'Household', 'Homeless', 'Transient', 'Institution', 'UrbanArea', 'TribalOrTraditionalArea', 'FarmArea', 'MalePopulation', 'Population', 'ProvinceContextYear', 'geography_level', 'geography_name', 'year', 'Food poverty headcount (FPL, %)', 'Gini coefficient (income per capita)', 'Poverty gap (P1, %)', 'Poverty headcount (P0, %)', 'Poverty share (%)', 'Severity of poverty (P2, %)', 'Dissatisfied with democracy', 'Distrust of national government', 'Distrust of political parties', 'Distrust of provincial government', 'Distrust of the IEC', 'QLFS_Q1_2024_Snapshot', 'qlfs_geography', 'QLFS_Q1_2024_DiscouragedWorkSeekers', 'QLFS_Q1_2024_Employed_thousand', 'QLFS_Q1_2024_AbsorptionRate', 'QLFS_Q1_

In [22]:
# Review the merge-generated columns before creating the final saved master table.
# We keep useful audit fields but remove temporary duplicate keys from the final dataset.

merge_columns = [
    "MunicipalityCode",
    "CurrentMunicipality",
    "Municipality_Group3",
    "ProvinceContextYear",
    "geography_level",
    "geography_name",
    "year"
]

print("Merge-related columns found:")
for column in merge_columns:
    print(f"- {column}: {'YES' if column in ward_master.columns else 'NO'}")

print("\nColumns that duplicate the original municipality name:")
print(
    ward_master[
        ["Municipality", "Municipality_Group3"]
    ]
    .dropna()
    .head(10)
    .to_string(index=False)
)

Merge-related columns found:
- MunicipalityCode: YES
- CurrentMunicipality: YES
- Municipality_Group3: YES
- ProvinceContextYear: YES
- geography_level: YES
- geography_name: YES
- year: YES

Columns that duplicate the original municipality name:
                  Municipality                 Municipality_Group3
ETH - eThekwini [Durban Metro] Ethekwini Metropolitan Municipality
ETH - eThekwini [Durban Metro] Ethekwini Metropolitan Municipality
ETH - eThekwini [Durban Metro] Ethekwini Metropolitan Municipality
ETH - eThekwini [Durban Metro] Ethekwini Metropolitan Municipality
ETH - eThekwini [Durban Metro] Ethekwini Metropolitan Municipality
ETH - eThekwini [Durban Metro] Ethekwini Metropolitan Municipality
ETH - eThekwini [Durban Metro] Ethekwini Metropolitan Municipality
ETH - eThekwini [Durban Metro] Ethekwini Metropolitan Municipality
ETH - eThekwini [Durban Metro] Ethekwini Metropolitan Municipality
ETH - eThekwini [Durban Metro] Ethekwini Metropolitan Municipality


In [23]:
# Remove temporary duplicate fields while keeping the keys needed to understand the final joins.
# The resulting table is the clean ward-year master dataset for turnout analysis and modelling.

columns_to_drop = [
    "Municipality_Group3",
    "geography_level",
    "geography_name",
    "year"
]

ward_master_final = ward_master.drop(
    columns=columns_to_drop
)

print("Final master shape:", ward_master_final.shape)

print("\nDropped columns:")
for column in columns_to_drop:
    print(f"- {column}")

print("\nFinal master columns:")
print(ward_master_final.columns.tolist())

Final master shape: (2599, 47)

Dropped columns:
- Municipality_Group3
- geography_level
- geography_name
- year

Final master columns:
['Province', 'Municipality', 'Ward', 'RegisteredVoters', 'SpoiltVotes', 'TotalValidVotes', 'VotingDistricts', 'VotingStations', 'ElectionYear', 'TurnoutRate', 'BoundaryConsistent', 'MunicipalityCode', 'CurrentMunicipality', 'Household', 'Homeless', 'Transient', 'Institution', 'UrbanArea', 'TribalOrTraditionalArea', 'FarmArea', 'MalePopulation', 'Population', 'ProvinceContextYear', 'Food poverty headcount (FPL, %)', 'Gini coefficient (income per capita)', 'Poverty gap (P1, %)', 'Poverty headcount (P0, %)', 'Poverty share (%)', 'Severity of poverty (P2, %)', 'Dissatisfied with democracy', 'Distrust of national government', 'Distrust of political parties', 'Distrust of provincial government', 'Distrust of the IEC', 'QLFS_Q1_2024_Snapshot', 'qlfs_geography', 'QLFS_Q1_2024_DiscouragedWorkSeekers', 'QLFS_Q1_2024_Employed_thousand', 'QLFS_Q1_2024_AbsorptionRa

Before saving, there is one important final check: the master contains current/snapshot context variables alongside historical election years. That's intentional in our pipeline, but we should document the coverage rather than silently treating those variables as election-year measurements.

Let's run the final integrity check for:

exact row count;
ward-year uniqueness;
key-field completeness;
BoundaryConsistent;
municipality-context coverage;
province-context coverage;
no accidental Group 4, 5, or 6 fields.

In [26]:
# Run the final integrity checks before saving the completed ward master dataset.
# This confirms the merged table is safe to use as the project's main analytical dataset.

expected_rows = 2599

print("Final rows:", len(ward_master_final))
print("Expected rows:", expected_rows)

print("\nWard-year duplicate keys:",
      ward_master_final.duplicated(
          subset=["Municipality", "Ward", "ElectionYear"]
      ).sum())

print("\nMissing key fields:")
print(
    ward_master_final[
        ["Province", "Municipality", "Ward", "ElectionYear", "TurnoutRate"]
    ]
    .isna()
    .sum()
)

print("\nBoundaryConsistent missing:",
      ward_master_final["BoundaryConsistent"].isna().sum())

print("\nMunicipality context rows:",
      ward_master_final["Population"].notna().sum())

print("Province context rows:",
      ward_master_final["Food poverty headcount (FPL, %)"].notna().sum())

excluded_fields = [
    "registered_voters",
    "Q13. Voting in the most recent national election",
    "StatsSA_Ward_Product_2022"
]

print("\nExcluded Group 5/6/4 fields present:")
for field in excluded_fields:
    print(f"- {field}: {'YES' if field in ward_master_final.columns else 'NO'}")

all_key_fields_complete = (
    ward_master_final[
        ["Province", "Municipality", "Ward", "ElectionYear", "TurnoutRate"]
    ]
    .notna()
    .all()
    .all()
)

if (
    len(ward_master_final) == expected_rows
    and ward_master_final.duplicated(
        subset=["Municipality", "Ward", "ElectionYear"]
    ).sum() == 0
    and all_key_fields_complete
    and ward_master_final["BoundaryConsistent"].isna().sum() == 0
    and ward_master_final["Population"].notna().sum() == 2489
    and ward_master_final["Food poverty headcount (FPL, %)"].notna().sum() == 2599
):
    print("\nFINAL GROUP 7 INTEGRITY CHECK: PASSED")
else:
    print("\nFINAL GROUP 7 INTEGRITY CHECK: REVIEW REQUIRED")

Final rows: 2599
Expected rows: 2599

Ward-year duplicate keys: 0

Missing key fields:
Province        0
Municipality    0
Ward            0
ElectionYear    0
TurnoutRate     0
dtype: int64

BoundaryConsistent missing: 0

Municipality context rows: 2489
Province context rows: 2599

Excluded Group 5/6/4 fields present:
- registered_voters: NO
- Q13. Voting in the most recent national election: NO
- StatsSA_Ward_Product_2022: NO

FINAL GROUP 7 INTEGRITY CHECK: PASSED


In [27]:
# Save the completed ward master dataset as the final Group 7 processed output.
# Reopening the file confirms that the saved dataset matches the validated table in memory.

ward_master_final.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

saved_master = pd.read_csv(
    output_path,
    encoding="utf-8-sig"
)

print("Output path:", output_path)
print("File exists:", output_path.exists())
print("Saved shape:", saved_master.shape)

print("\nSaved columns:", len(saved_master.columns))
print("Saved rows:", len(saved_master))

print("\nSaved ward-year duplicate keys:",
      saved_master.duplicated(
          subset=["Municipality", "Ward", "ElectionYear"]
      ).sum())

if (
    output_path.exists()
    and saved_master.shape == ward_master_final.shape
    and saved_master.columns.tolist() == ward_master_final.columns.tolist()
    and saved_master.duplicated(
        subset=["Municipality", "Ward", "ElectionYear"]
    ).sum() == 0
):
    print("\nGROUP 7 SAVE VALIDATION: PASSED")
else:
    print("\nGROUP 7 SAVE VALIDATION: REVIEW REQUIRED")

Output path: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\07_ward_master_dataset\ward_master_dataset_2011_2021.csv
File exists: True
Saved shape: (2599, 47)

Saved columns: 47
Saved rows: 2599

Saved ward-year duplicate keys: 0

GROUP 7 SAVE VALIDATION: PASSED
